In [26]:
import time
import torch
import pandas as pd
import psutil
import threading

In [2]:
from transformers import (
    pipeline,
    MarianMTModel, MarianTokenizer,
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoTokenizer, AutoModelForSeq2SeqLM
)

c:\Users\majd\Desktop\pfe project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [49]:
# test_bubbles = [
#     "ちっくしょ〜",
#     "脅迫じゃねーかあのパワハラ教師！！",
#     "うぉぉぉおぉおおおぉ！？",
#     "ん？",
#     "失礼しまーす",
#     "すっすみません今散らかってて",
#     "ようこそ！",
#     "風船いっばい夢いっぱいの天文部へ！",
#     "な、なんだこれ！？",
#     "．．．風船？"
# ]
# test_bubbles=[
#     "マジで最悪だわ…",
#     "えええええ！？ うそだろおおお！！",
#     "ちょっと待ってくれよぉ！！",
#     "あははははっ！ やられたぁ！",
#     "……バカ",
#     "てめぇら全員まとめてぶっ飛ばすぞ！！",
#     "ふぁぁ眠い",
#     "好きだよ…ずっと前から…",
#     "うわっ！ 熱っ！ 熱すぎるって！！",
#     "もう…やめてくれないかな…",
#     " 最高じゃねぇか！！",
#     "……ありがとう",
# ]
test_bubbles=[
    "風船が割れてパラシュートで落ちてきて．．．",
    "カメラを回収したら地球の写真が撮れてるんです！",
    "以前テレビで見たんです！",
    "風船でブワフワとカメラが宇宙へ行って．．",
    "ね？すっごくワクワクしませんか？",
    "それに．．．",
    "だからたくさん風船がいるんですっ",
    "面倒くっさい奴人に押しつけてきやがって．．．",
    "なにがちょっとした手伝いだあのボンクラ教師！！",
    "相川さんもいっぱい膨らませちゃってください！",
    " あのなぁ",
]

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", dtype="auto", attn_implementation="sdpa")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 520.52it/s, Materializing param=model.shared.weight]                                   
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [6]:
def get_gpu_memory():
    if device == "cuda":
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 ** 3)  # GB
    return 0.0

In [23]:
def monitor_ram(peak_ram):
    """Runs in background thread and updates peak RAM every 0.1s"""
    while not peak_ram["done"]:
        current = psutil.Process().memory_info().rss / (1024 ** 3)  # GB
        if current > peak_ram["max"]:
            peak_ram["max"] = current
        time.sleep(0.1)

In [7]:
def translate_ja_to_ar(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        src_lang="jpn_Jpan"
    )

    arb_id = tokenizer.convert_tokens_to_ids("arb_Arab")

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=arb_id,
        max_length=256
    )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

In [50]:
print("\n=== Loading NLLB-600M ===")


torch.cuda.reset_peak_memory_stats() if device == "cuda" else None
peak_ram_nllb = {"max": 0.0, "done": False}
monitor_thread = threading.Thread(target=monitor_ram, args=(peak_ram_nllb,))
monitor_thread.start()
start = time.perf_counter()

for text in test_bubbles:
    print(text)
    _ = translate_ja_to_ar(text)
    print(_)

nllb_time = time.perf_counter() - start
nllb_mem = get_gpu_memory()
peak_ram_nllb["done"] = True
monitor_thread.join()

print(f"NLLB-600M: {nllb_time:.3f} s total | {nllb_time/len(test_bubbles):.3f} s per sentence | {nllb_mem:.2f} GB peak,Peak RAM: {peak_ram_nllb['max']:.2f} GB")


=== Loading NLLB-600M ===
風船が割れてパラシュートで落ちてきて．．．
وكانت الهواء قد انفجر وقع على مقصورة على الهواء
カメラを回収したら地球の写真が撮れてるんです！
عندما يتم استرجاع الكاميرا، يتم تصوير الأرض!
以前テレビで見たんです！
لقد رأيتها في التلفاز من قبل!
風船でブワフワとカメラが宇宙へ行って．．
في الزهور، ذهب بوفوا وكاميرا إلى الفضاء.
ね？すっごくワクワクしませんか？
ألا تشعرين بالتحسن؟
それに．．．
وبالإضافة إلى ذلك..
だからたくさん風船がいるんですっ
لهذا السبب يوجد الكثير من الحمامات.
面倒くっさい奴人に押しつけてきやがって．．．
ويدفع العبيد المزعومين على الجهد ويدفعهم إلى الجهد.
なにがちょっとした手伝いだあのボンクラ教師！！
كم كان ذلك المعلم البونكرا الذي ساعدني قليلاً!!
相川さんもいっぱい膨らませちゃってください！
واكثر من ذلك، فليفُضَّت ساكاوا أيضاً!
 あのなぁ
وذلك بسبب ذلك.
NLLB-600M: 50.300 s total | 4.573 s per sentence | 0.00 GB peak,Peak RAM: 2.33 GB


In [9]:
# ─────────────────────────────────────────────────────────────
# 2. M2M100-418M
# ─────────────────────────────────────────────────────────────
print("\n=== Loading M2M100-418M ===")
m2m_tokenizer = M2M100Tokenizer.from_pretrained("models/m2m100_418M")
m2m_model = M2M100ForConditionalGeneration.from_pretrained("models/m2m100_418M").to(device)




=== Loading M2M100-418M ===


Loading weights: 100%|██████████| 512/512 [00:01<00:00, 416.29it/s, Materializing param=model.shared.weight]                                   
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [18]:
def translate_ja_to_ar_m2m100(japanese_text: str, max_length: int = 200) -> str:
    m2m_tokenizer.src_lang = "ja"      
    m2m_tokenizer.tgt_lang = "ar"
    inputs = m2m_tokenizer(japanese_text, return_tensors="pt")
    generated_tokens = m2m_model.generate(
        **inputs,
        forced_bos_token_id=m2m_tokenizer.get_lang_id("ar"),
        num_beams=5,
        early_stopping=True
    )
    translated = m2m_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0].strip()
    return translated

In [51]:
torch.cuda.reset_peak_memory_stats() if device == "cuda" else None
peak_ram_m2m = {"max": 0.0, "done": False}
monitor_thread = threading.Thread(target=monitor_ram, args=(peak_ram_m2m,))
monitor_thread.start()
start = time.perf_counter()
for text in test_bubbles:
    print(text)
    _ = translate_ja_to_ar_m2m100(text)
    print(_)

m2m_time = time.perf_counter() - start
m2m_mem = get_gpu_memory()
peak_ram_m2m["done"] = True
monitor_thread.join()

print(f"M2M100-418M: {m2m_time:.3f} s total | {m2m_time/len(test_bubbles):.3f} s per sentence | {m2m_mem:.2f} GB | peakPeak RAM: {peak_ram_m2m['max']:.2f} GB")

風船が割れてパラシュートで落ちてきて．．．
لقد سقطت البالونات، وسقطت في المظلة..
カメラを回収したら地球の写真が撮れてるんです！
إذا قمت بتصوير الكاميرا ، فستقوم بتصوير الأرض!
以前テレビで見たんです！
لقد رأيته في التلفزيون من قبل!
風船でブワフワとカメラが宇宙へ行って．．
على متن البالونات و الكاميرا في الفضاء..
ね？すっごくワクワクしませんか？
هل أنت متحمس للغاية؟
それに．．．
وبالإضافة إلى...
だからたくさん風船がいるんですっ
وهناك الكثير من الباليه.
面倒くっさい奴人に押しつけてきやがって．．．
أضغط على الرجل المضطرب و أضغط على...
なにがちょっとした手伝いだあのボンクラ教師！！
ما هي المساعدة التي يقدمها هذا المعلم البنكرياس!
相川さんもいっぱい膨らませちゃってください！
أهلا وسهلا يا أهلا وسهلا يا أهلا وسهلا!
 あのなぁ
هذا هو
M2M100-418M: 42.109 s total | 3.828 s per sentence | 0.00 GB | peakPeak RAM: 3.68 GB


In [12]:
# ─────────────────────────────────────────────────────────────
# 3. Opus-MT pivot (ja-en + en-ar)
# ─────────────────────────────────────────────────────────────
print("\n=== Loading Opus-MT ja-ar ===")
# Load the Japanese → Arabic model
model_name = "Helsinki-NLP/opus-mt-ja-ar"
opus_tokenizer = MarianTokenizer.from_pretrained(model_name)
opus_model = MarianMTModel.from_pretrained(model_name)


=== Loading Opus-MT ja-ar ===


c:\Users\majd\Desktop\pfe project\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 404.97it/s, Materializing param=model.shared.weight]                                  
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [44]:
def translate_ja_to_ar_opus(text):    
     inputs = opus_tokenizer(text, return_tensors="pt", padding=True)
    
    
     translated_ids = opus_model.generate(
        **inputs,
        max_length=200,
        num_beams=4,          # optional: improves quality a bit
        early_stopping=True
    )
    
     return opus_tokenizer.decode(translated_ids[0], skip_special_tokens=True).strip()

In [52]:
torch.cuda.reset_peak_memory_stats() if device == "cuda" else None
peak_ram_opus = {"max": 0.0, "done": False}
monitor_thread = threading.Thread(target=monitor_ram, args=(peak_ram_opus,))
monitor_thread.start()
start = time.perf_counter()
for text in test_bubbles:
    print(text)
    _ = translate_ja_to_ar_opus(text)
    print(_)

opus_time = time.perf_counter() - start
opus_mem = get_gpu_memory()
peak_ram_opus["done"] = True
monitor_thread.join()

print(f"opus-mt: {opus_time:.3f} s total | {opus_time/len(test_bubbles):.3f} s per sentence | {opus_mem:.2f} GB peak| Peak RAM: {peak_ram_opus['max']:.2f} GB")

風船が割れてパラシュートで落ちてきて．．．
لقد إنكسر البالون وسقط بالمظلة
カメラを回収したら地球の写真が撮れてるんです！
إنّهم يلتقطون صوراً للأرض عندما نحصل على الكاميرا!
以前テレビで見たんです！
لقد رأيتك على التلفاز من قبل
風船でブワフワとカメラが宇宙へ行って．．
-بالونة، والكاميرات تذهب للفضاء .
ね？すっごくワクワクしませんか？
هل انت متحمس جداً ؟
それに．．．
و...
だからたくさん風船がいるんですっ
هناك الكثير من البالونات هناك
面倒くっさい奴人に押しつけてきやがって．．．
لقد وضعتني في مأزق كبير
なにがちょっとした手伝いだあのボンクラ教師！！
ماذا يمكنني أن أفعل لك؟
相川さんもいっぱい膨らませちゃってください！
سيّدي، سيّدي، سيّدي!
 あのなぁ
مرحباً.
opus-mt: 5.268 s total | 0.479 s per sentence | 0.00 GB peak| Peak RAM: 3.56 GB


In [32]:
results = {
    "Model": ["NLLB-600M", "M2M100-418M", "Opus-MT pivot"],
    "Total time (10 sentences)": [f"{nllb_time:.3f} s", f"{m2m_time:.3f} s", f"{opus_time:.3f} s"],
    "Time per sentence": [f"{nllb_time/10:.3f} s", f"{m2m_time/10:.3f} s", f"{opus_time/10:.3f} s"],
    "Peak RAM during inference": [f"{peak_ram_nllb['max']:.2f} GB", f"{peak_ram_m2m['max']:.2f} GB", f"{peak_ram_opus['max']:.2f} GB"]
}

df = pd.DataFrame(results)
print("\n" + "="*70)
print("CPU + RAM Comparison Summary (10 sentences)")
print("="*70)
print(df.to_string(index=False))


CPU + RAM Comparison Summary (10 sentences)
        Model Total time (10 sentences) Time per sentence Peak RAM during inference
    NLLB-600M                  47.016 s           4.702 s                   2.32 GB
  M2M100-418M                  31.221 s           3.122 s                   3.40 GB
Opus-MT pivot                   4.447 s           0.445 s                   3.17 GB
